# Phase 2 — exposure-augmentation experiment `generalized_lol_v2_expaug` (Colab GPU)

Identical to the Phase-2 run except: input-only exposure augmentation on training inputs (p 0.5, log-uniform [0.5, 3.0], capped at 1/q99.9 of max-RGB). Output: `experiments/generalized_lol_v2_expaug/`. The Phase-2 run in `experiments/generalized_lol_v2/` is never written.

Fixed protocol (see `experiments/generalized_lol_v2/README.md`):
- same U-Net / L1 / Adam 2e-4 / crop 256 / hflip / seed 42 as the LOL-v1 baseline
- every batch = 2 LOL-v2-real + 2 LOL-v2-synthetic; 112 steps x 50 epochs = 5,600 steps
- best checkpoint = max (real_val_PSNR + synthetic_val_PSNR)/2 on scene-disjoint val splits
- test once: LOL-v2-real Test, LOL-v2-synthetic Test (seen); SICE 58 sequences, LoLI-Street Val (unseen)

**Runtime → Change runtime type → GPU.**

Expected Drive layout (upload from the Mac project, WITHOUT `.venv/`, `data/`, extracted `data_prepared/phase2/`):
```
MyDrive/LowLight-Generalization/
├── configs/  scripts/  src/  notebooks/  requirements.txt
├── experiments/generalized_lol_v2/splits/      (lol_v2_real_split.json, lol_v2_synthetic_split.json, sice_test_selection.json)
└── data_prepared/phase2.zip                    (2.02 GB, byte-identical copies + manifest.json)
```

## 1. GPU

In [ ]:
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No CUDA GPU: Runtime -> Change runtime type -> GPU'
print('device count', torch.cuda.device_count(), '| current', torch.cuda.current_device(), '|', torch.cuda.get_device_name(0))
!nvidia-smi

## 2. Drive + project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT_DIR = '/content/drive/MyDrive/LowLight-Generalization'
for p in ['configs/generalized_lol_v2.yaml', 'configs/generalized_lol_v2_expaug.yaml', 'src/datasets/exposure_augment.py', 'scripts/smoke_test_expaug.py', 'scripts/train_generalized.py', 'src/datasets/phase2.py',
          'experiments/generalized_lol_v2/splits/lol_v2_real_split.json',
          'experiments/generalized_lol_v2/splits/lol_v2_synthetic_split.json',
          'experiments/generalized_lol_v2/splits/sice_test_selection.json', 'data_prepared/phase2.zip']:
    assert os.path.exists(os.path.join(PROJECT_DIR, p)), f'Missing on Drive: {p}'
%cd {PROJECT_DIR}

## 3. Environment (torch preinstalled on Colab; only LPIPS added)

In [ ]:
!pip -q install 'lpips>=0.1.4'
import importlib
for m in ['torch', 'torchvision', 'PIL', 'numpy', 'matplotlib', 'yaml', 'lpips']:
    mod = importlib.import_module(m); print(f'{m:12s}', getattr(mod, '__version__', '?'))

## 4. Data to local disk (fast I/O); archive on Drive is not modified

In [ ]:
DATA_ROOT = '/content/phase2'
if not os.path.exists(os.path.join(DATA_ROOT, 'manifest.json')):
    !unzip -q {PROJECT_DIR}/data_prepared/phase2.zip -d /content
!ls {DATA_ROOT}

## 5. Re-verify splits reproduce identically + smoke test (stops on any failure)

In [ ]:
!python scripts/make_phase2_splits.py --data-root {DATA_ROOT} 2>&1 | grep -v '^  SICE'
!python scripts/smoke_test_expaug.py --data-root {DATA_ROOT} --device cuda --workers 0 2 --skip-snapshot-check 2>&1 | tail -45

## 6. Train (5,600 steps). Integrity checks run first; training refuses to overwrite an existing run.
If Colab disconnects: set `RESUME = True` and re-run cells 1, 2, 3, 4 and this one.

In [ ]:
RESUME = False
EXP = os.path.join(PROJECT_DIR, 'experiments/generalized_lol_v2_expaug')
last = os.path.join(EXP, 'checkpoints/last.pth')
os.makedirs(os.path.join(EXP, 'logs'), exist_ok=True)
if RESUME:
    assert os.path.exists(last), 'nothing to resume'
    !python scripts/train_generalized.py --config configs/generalized_lol_v2_expaug.yaml --data-root {DATA_ROOT} --device cuda --resume {last} 2>&1 | tee -a {EXP}/logs/train_log.txt
else:
    assert not os.path.exists(last), 'A previous run exists; move it away or set RESUME=True'
    !python scripts/train_generalized.py --config configs/generalized_lol_v2_expaug.yaml --data-root {DATA_ROOT} --device cuda 2>&1 | tee {EXP}/logs/train_log.txt

## 7. Check training completed

In [ ]:
import json
s = json.load(open(os.path.join(EXP, 'logs/train_summary.json')))
print(json.dumps(s, indent=2))
assert s['epochs_completed'] == 50 and s['optimizer_steps_total'] == 5600 and not s['pipeline_test_mode']
for f in ['best.pth', 'last.pth', 'final.pth']:
    assert os.path.exists(os.path.join(EXP, 'checkpoints', f)), f

## 8. Evaluate the best checkpoint ONCE on all test domains (refuses to run twice)

In [ ]:
!python scripts/evaluate_generalized.py --config configs/generalized_lol_v2_expaug.yaml --data-root {DATA_ROOT} --device cuda 2>&1 | tee {EXP}/logs/eval_log.txt
!python scripts/plot_generalized_curves.py --config configs/generalized_lol_v2_expaug.yaml

## 9. Results

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open(os.path.join(EXP, 'evaluation/metrics_summary.md')).read()))
display(Image(os.path.join(EXP, 'evaluation/training_curves.png')))
for d in ['lol_v2_real_test', 'lol_v2_synthetic_test', 'sice', 'loli_street_val']:
    display(Image(os.path.join(EXP, f'evaluation/qualitative/{d}.png')))
!grep -i -E "warn|error|traceback" {EXP}/logs/train_log.txt | sort | uniq -c | head || true